# Trajectory Pilot — Post-hoc Stratified Evaluation

对 `run_trajectory_pilot.ipynb` 产出的 3 个模态 run（V / V+T / V+K+T）分别做事件锁相分层评估。

**关键判读**：

| 事件附近 CCC | 空窗期 CCC | 事件附近 pred_std | 解读 |
|---|---|---|---|
| >0.1 | ≈0 | >0.1 | **模型在关键区有信号**，范式可救 |
| ≈0 | ≈0 | ≈0 | 模型全程保守（输出平直），任务太难 |
| ≈0 | ≈0 | >0.1 | **模型在乱输出**，信号学反了 |
| <0 | ≈0 | >0.1 | 训练/推理分布 gap 严重 |

**产出**：每个 run 目录下的 `stratified_eval.json` + 汇总表。

In [1]:
# Cell 1: Mount + cd
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ProjectExperiment
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Mounted at /content/drive
/content/drive/MyDrive/ProjectExperiment
NVIDIA L4, 23034 MiB


In [2]:
# Cell 2: Locate the 3 pilot runs — pick latest timestamped dir THAT HAS ckpt_best.pt
import glob
from pathlib import Path

BASE = '/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed'

run_dirs = {}
for mod_tag in ['single_video', 'dual_video_telem', 'triple_video_km_telem']:
    matches = sorted(glob.glob(f'{BASE}/{mod_tag}/*seed0*'))
    # filter: must contain ckpt_best.pt (skip empty/aborted dirs)
    valid = [m for m in matches if (Path(m) / 'ckpt_best.pt').exists()]
    assert valid, f'no completed run dir with ckpt_best.pt for {mod_tag}; candidates: {matches}'
    run_dirs[mod_tag] = valid[-1]    # latest *completed* run
    skipped = [m for m in matches if m not in valid]
    if skipped:
        print(f'{mod_tag}: skipped {len(skipped)} empty dir(s): {[Path(s).name for s in skipped]}')
    print(f'{mod_tag}: {run_dirs[mod_tag]}')

single_video: skipped 2 empty dir(s): ['2026-04-23_22-16-16__amucs_trajectory__eft__video__seed0', '2026-04-23_22-18-09__amucs_trajectory__eft__video__seed0']
single_video: /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed/single_video/2026-04-23_19-30-36__amucs_trajectory__eft__video__seed0
dual_video_telem: skipped 1 empty dir(s): ['2026-04-24_08-14-48__amucs_trajectory__eft__telem_video__seed0']
dual_video_telem: /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed/dual_video_telem/2026-04-23_19-35-24__amucs_trajectory__eft__telem_video__seed0
triple_video_km_telem: /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed/triple_video_km_telem/2026-04-23_19-43-54__amucs_trajectory__eft__km_telem_video__seed0


In [3]:
# Cell 3: Run stratified eval for each modality — stream stdout+stderr to notebook
import subprocess, sys

for mod_tag, run_dir in run_dirs.items():
    print(f'\n{"="*70}')
    print(f'Evaluating {mod_tag}')
    print(f'{"="*70}', flush=True)
    proc = subprocess.Popen(
        [sys.executable, '-u', 'scripts/evaluate_trajectory.py',
         '--run_dir', run_dir, '--splits', 'val', 'test'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f'\n!!! {mod_tag} exited with code {proc.returncode} — continuing to next')


Evaluating single_video
[load] config from /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed/single_video/2026-04-23_19-30-36__amucs_trajectory__eft__video__seed0/config.yaml
[load] checkpoint from /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed/single_video/2026-04-23_19-30-36__amucs_trajectory__eft__video__seed0/ckpt_best.pt
[AMuCSTrajectoryDataModule] train-set Δσ = 18.4468
[val] collecting predictions ...
[val] 6259 anchors  pred_shape=(6259, 51)  target_shape=(6259, 51)
[val] strata: event=4999 (79.9%) quiet=1260 (20.1%)
[test] collecting predictions ...
[test] 13493 anchors  pred_shape=(13493, 51)  target_shape=(13493, 51)
[test] strata: event=9966 (73.9%) quiet=3527 (26.1%)

=== VAL ===
stratum        n      mse     ccc pred_std  gt_std  peak_f1  lead_s  amp_err  ev_corr
-------------------------------------------------------------------------------------
event       4999

In [4]:
# Cell 4: Cross-modality comparison
import json

print(f"{'modality':<24} {'split':<6} {'stratum':<10} {'n':>6} {'mse':>8} {'ccc':>8} {'pred_std':>9} {'peak_f1':>8} {'ev_corr':>9}")
print('-' * 100)

for mod_tag, run_dir in run_dirs.items():
    ev_path = Path(run_dir) / 'stratified_eval.json'
    if not ev_path.exists():
        print(f'{mod_tag}: no stratified_eval.json')
        continue
    ev = json.load(open(ev_path))
    for split_name, strata in ev['results'].items():
        for stratum in ['event', 'quiet', 'overall']:
            m = strata.get(stratum, {})
            if m.get('n', 0) == 0:
                continue
            def fmt(v, w=8, d=4):
                if v is None or (isinstance(v, float) and v != v):   # NaN
                    return f'{"nan":>{w}}'
                return f'{v:>{w}.{d}f}' if isinstance(v, float) else f'{v:>{w}}'
            print(f"{mod_tag:<24} {split_name:<6} {stratum:<10} "
                  f"{m.get('n',0):>6d} {fmt(m.get('mse'))} {fmt(m.get('ccc'))} "
                  f"{fmt(m.get('pred_std'),9)} {fmt(m.get('peak_f1'))} {fmt(m.get('event_corr_mean'),9)}")
    print()

modality                 split  stratum         n      mse      ccc  pred_std  peak_f1   ev_corr
----------------------------------------------------------------------------------------------------
single_video             val    event        4999   1.5240  -0.0007    0.0606      nan    0.0270
single_video             val    quiet        1260   1.3953  -0.0026    0.0668      nan   -0.0105
single_video             val    overall      6259   1.4981  -0.0010    0.0619      nan    0.0194
single_video             test   event        9966   2.3903  -0.0011    0.0627      nan    0.0260
single_video             test   quiet        3527   2.6355   0.0002    0.0646      nan    0.0283
single_video             test   overall     13493   2.4544  -0.0007    0.0632      nan    0.0266

dual_video_telem         val    event        4981   1.5081   0.0009    0.0633      nan    0.0412
dual_video_telem         val    quiet        1156   1.2982  -0.0010    0.0744      nan    0.0498
dual_video_telem         